# Render del logo cinematográfico con GPU gratis (Colab o Kaggle)

1. **Colab:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.
   **Kaggle:** Settings → Accelerator → **GPU T4 x2** (y activar *Internet*).
2. Ejecutá las celdas en orden (o *Run all*). En la celda 3 subís `logo.png` y `logo_cine.py`.
3. Al final se descarga `logo_cine.mp4`.

Con GPU cada frame tarda pocos segundos, así que podés subir la calidad (`SAMPLES`).

In [ ]:
# 1) Verificar la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Descargar Blender 5.2 LTS e instalar las librerías que usa el script
BLENDER_VER = "5.2.3"
!wget -q https://download.blender.org/release/Blender5.2/blender-{BLENDER_VER}-linux-x64.tar.xz
!tar -xf blender-{BLENDER_VER}-linux-x64.tar.xz && rm blender-{BLENDER_VER}-linux-x64.tar.xz
BL = f"/content/blender-{BLENDER_VER}-linux-x64" if __import__("os").path.exists("/content") else f"blender-{BLENDER_VER}-linux-x64"
import glob
PY = glob.glob(f"{BL}/5.2/python/bin/python3*")[0]
!{PY} -m ensurepip -q
!{PY} -m pip install -q pillow scipy scikit-image
!{BL}/blender --version | head -1

In [ ]:
# 3) Subir logo.png y logo_cine.py
import os, shutil
try:
    from google.colab import files          # Colab
    files.upload()
except ImportError:                         # Kaggle: agregalos como dataset (Add data) y se copian acá
    for p in glob.glob("/kaggle/input/*/*"):
        if p.endswith(("logo.png", "logo_cine.py")):
            shutil.copy(p, ".")
print(os.listdir("."))

In [ ]:
# 4) Renderizar con GPU (OptiX/CUDA). Más SAMPLES = menos ruido.
SAMPLES = 128
RES_PCT = 100          # 150 = 1920x1080
!mkdir -p frames
!{BL}/blender -b -P logo_cine.py -- --logo logo.png --out frames/ --gpu --samples {SAMPLES} --res-pct {RES_PCT} 2>&1 | grep -E "Saved|Error|Traceback" 

In [ ]:
# 5) Armar el video y descargarlo
!ffmpeg -loglevel error -y -framerate 24 -i frames/%04d.png -c:v libx264 -pix_fmt yuv420p -crf 16 -movflags +faststart logo_cine.mp4
try:
    from google.colab import files
    files.download("logo_cine.mp4")
except ImportError:
    shutil.copy("logo_cine.mp4", "/kaggle/working/logo_cine.mp4")   # pestaña Output
    print("Listo: /kaggle/working/logo_cine.mp4")